In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from noise_analysis_utils import load_fake_detections_fits, get_detections

In [ ]:
#fake_detections_file = "./analysis_pairs/threshold_analysis_results_rec1_rec50000.fits"
fake_detections_file = "threshold_analysis_rec1_rec5.fits" # testing purposes

In [ ]:
# read the fake detections from the FITS file
meta, det_offsets, positions, energies = load_fake_detections_fits(fake_detections_file)

# combination grid used when the file was written, recovered from the metadata itself.
# WINDOW/OFFSET/THRESHOLD were always iterated in ascending order when the file was
# written, so np.unique's ascending output reproduces that same order.
windows = np.unique(meta["WINDOW"]).tolist()
offsets = np.unique(meta["OFFSET"]).tolist()
th_sigmas = np.unique(meta["THRESHOLD"]).tolist()
nnrecords = len(np.unique(meta["RECORD"]))

# reconstruct the per-combination detection lists, in the same (record, window, offset,
# threshold) row-major order they were written in (rows preserve that order in the FITS table)
fake_positions_list = [positions[det_offsets[i]:det_offsets[i + 1]] for i in range(len(meta))]
fake_energies_list = [energies[det_offsets[i]:det_offsets[i + 1]] for i in range(len(meta))]

In [ ]:
# calculate mean and std of the number of fake detections across all records, for each combination
# of window, offset, and threshold. Number of detections per combo is just len() of its stored
# positions array, so this is derived straight from fake_positions_list -- no separate per-record
# false_detections array needs to be tracked during the main loop.
n_detections_flat = np.array([len(p) for p in fake_positions_list], dtype=float)

valid_mask = np.array([[not (window == 0 and offset > 0) for offset in offsets] for window in windows])
n_detections_grid = n_detections_flat.reshape(nnrecords, valid_mask.sum(), len(th_sigmas))

mean_false_detections = np.full((len(windows), len(offsets), len(th_sigmas)), np.nan)
std_false_detections = np.full((len(windows), len(offsets), len(th_sigmas)), np.nan)
mean_false_detections[valid_mask] = n_detections_grid.mean(axis=0)
std_false_detections[valid_mask] = n_detections_grid.std(axis=0)

## Plot False detections vs Threshold for each w/o

In [ ]:
# plot mean and std of false detections for each combination of window, offset, and threshold
# single axis (so an iso-FAR curve can be overlaid later):
#   - window -> continuous sequential colormap + colorbar (avoids one legend entry per window)
#   - offset -> fixed marker shape, with its own small legend (few distinct offsets)
markers = ['o', 's', '^', 'D', 'v', 'P', 'X', '*', '<', '>']
offset_marker = {offset: markers[i % len(markers)] for i, offset in enumerate(offsets)}

non_baseline_windows = [w for w in windows if w > 0]
cmap = plt.cm.viridis
norm = plt.Normalize(vmin=min(non_baseline_windows), vmax=max(non_baseline_windows))

fig, ax = plt.subplots(figsize=(10, 6))
for iw, window in enumerate(windows):
    for io, offset in enumerate(offsets):
        if window == 0 and offset > 0:
            continue  # skip this case since it is not defined
        if window == 0 and offset == 0:
            color = 'black'
            linestyle = '--'
        else:
            color = cmap(norm(window))
            linestyle = '-'
        ax.errorbar(
            th_sigmas,
            mean_false_detections[iw, io, :],
            # use as error bar the error of the mean, which is std/sqrt(N), where N is the number of records
            yerr=std_false_detections[iw, io, :] / np.sqrt(nnrecords),
            capsize=2,
            marker=offset_marker[offset],
            markersize=5,
            linestyle=linestyle,
            linewidth=1.3,
            color=color,
            alpha=0.9,
        )
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label("Window (samples)")

offset_handles = [
    plt.Line2D([0], [0], color='gray', marker=offset_marker[offset], linestyle='None',
                markersize=6, label=f"Offset {offset}")
    for offset in offsets
]
offset_handles.append(
    plt.Line2D([0], [0], color='black', linestyle='--', marker='o', markersize=5,
                label="Window=0, Offset=0 (baseline)")
)
ax.legend(handles=offset_handles, title="Offset", loc='upper right', fontsize=8, ncol=2)

ax.set_xlabel("Threshold (sigmas)")
ax.set_ylabel("Mean number of false detections")
ax.set_title(f"False Detections vs Threshold for {nnrecords} Noise Records")
ax.grid(alpha=0.3)
plt.show()